# DCGAN Approach
- Ref paper: https://arxiv.org/pdf/2208.05593
- Paper summary: This paper addresses the severe class imbalance in diabetic retinopathy datasets by using a Deep Convolutional Generative Adversarial Network (DCGAN) to synthesize images of underrepresented, severe disease stages. To evaluate the generated imagery, the researchers determined that Fréchet Inception Distance (FID) is the best metric for measuring visual quality, while Multi-Scale Structural Similarity Index (MS-SSIM) and Cosine Distance (CD) are optimal for assessing dataset diversity. Moreover, augmenting the training data with these synthetic images significantly improved the diagnostic performance compared to models trained solely on the original, imbalanced data.

# Set Up

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess

def run(cmd):
    subprocess.run(cmd, shell=True, check=True)

# Copy dataset from Drive to local SSD
run("cp '/content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/archive.zip' /content/RFMiD.zip")
run("unzip -q /content/RFMiD.zip -d /content/RFMiD")

# Dependencies
run("pip install -q torch torchvision clean-fid scipy pandas Pillow tqdm matplotlib")

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
# print(f"GPU     : {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
PyTorch : 2.10.0+cu128
CUDA    : True


In [2]:
def run(cmd):
    subprocess.run(cmd, shell=True, check=True, text=True)

# Delete the entire __MACOSX directory
run("rm -rf /content/RFMiD/__MACOSX")

# Find and delete all hidden .DS_Store files recursively
run('find /content/RFMiD -name ".DS_Store" -delete')

print("Mac junk removed successfully!")

Mac junk removed successfully!


In [3]:
import os, pandas as pd
BASE = "/content/RFMiD"

# Walk the unzipped folder to understand layout
for root, dirs, files in os.walk(BASE):
    level = root.replace(BASE, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files[:5]:
            print(f"  {indent}{f}")

RFMiD/
  archive/
    Training_Set/
      Training/
    Evaluation_Set/
      Validation/
    Test_Set/
      Test/


# Label Mapping

- We selected Retinal Fundus Multi-disease image Dataset (RFMiD) for experimentation in this project.
- Class Distribution of normal 669 images and diseased of 2531 images. (Count by 'Disease_Risk' column), but since the labels and images are separate from one another, we perform the mapping first.
- The data has already been resized to 512 x 512 and center cropped.

In [4]:
import os, shutil, pandas as pd
from pathlib import Path
from tqdm import tqdm

BASE       = "/content/RFMiD"
NORMAL_DIR  = "/content/data/normal"
ABNORMAL_DIR = "/content/data/abnormal"

os.makedirs(NORMAL_DIR,   exist_ok=True)
os.makedirs(ABNORMAL_DIR, exist_ok=True)

EXTENSIONS = [".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG"]

def find_dataset_paths(base):
    """Auto-detect CSV and image folder paths regardless of zip layout."""
    paths = {}
    for root, dirs, files in os.walk(base):
        for f in files:
            name = f.lower()
            if name.endswith(".csv"):
                if "train" in name:   paths["train_csv"] = os.path.join(root, f)
                elif "val"   in name: paths["val_csv"]   = os.path.join(root, f)
                elif "test"  in name: paths["test_csv"]  = os.path.join(root, f)
        imgs = [f for f in files if any(f.endswith(e) for e in EXTENSIONS)]
        if imgs:
            folder = os.path.basename(root).lower()
            if "train" in folder:   paths["train_img"] = root
            elif "val"   in folder: paths["val_img"]   = root
            elif "test"  in folder: paths["test_img"]  = root
    print("Detected paths:")
    for k, v in paths.items():
        print(f"  {k:12s} → {v}")
    return paths

def resolve(img_id, img_dir):
    for ext in EXTENSIONS:
        p = os.path.join(img_dir, f"{img_id}{ext}")
        if os.path.exists(p):
            return p
    return None

def load_split(csv_key, img_key, paths):
    if csv_key not in paths or img_key not in paths:
        return pd.DataFrame()
    df = pd.read_csv(paths[csv_key])
    df["img_path"] = df["ID"].apply(lambda x: resolve(x, paths[img_key]))
    df["split"]    = csv_key.replace("_csv", "")
    found = df["img_path"].notna().sum()
    print(f"  {csv_key}: {len(df)} rows | {found} images found | "
          f"{len(df)-found} missing")
    return df

In [5]:
paths    = find_dataset_paths(BASE)
train_df = load_split("train_csv", "train_img", paths)
val_df   = load_split("val_csv",   "val_img",   paths)
test_df  = load_split("test_csv",  "test_img",  paths)

all_df      = pd.concat([train_df, val_df, test_df], ignore_index=True)
normal_df   = all_df[(all_df["Disease_Risk"] == 0) & all_df["img_path"].notna()]
abnormal_df = all_df[(all_df["Disease_Risk"] == 1) & all_df["img_path"].notna()]

print(f"\nNormal   : {len(normal_df)}")
print(f"Abnormal : {len(abnormal_df)}")

Detected paths:
  train_csv    → /content/RFMiD/archive/Training_Set/RFMiD_Training_Labels.csv
  train_img    → /content/RFMiD/archive/Training_Set/Training
  val_csv      → /content/RFMiD/archive/Evaluation_Set/RFMiD_Validation_Labels.csv
  val_img      → /content/RFMiD/archive/Evaluation_Set/Validation
  test_csv     → /content/RFMiD/archive/Test_Set/RFMiD_Testing_Labels.csv
  test_img     → /content/RFMiD/archive/Test_Set/Test
  train_csv: 1920 rows | 1920 images found | 0 missing
  val_csv: 640 rows | 640 images found | 0 missing
  test_csv: 640 rows | 640 images found | 0 missing

Normal   : 669
Abnormal : 2531


In [6]:
def copy_class(df, out_dir, label):
    copied = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Copying {label}"):
        # Create a unique filename (e.g., "train_1.png", "val_1.png")
        orig_name = os.path.basename(row["img_path"])
        unique_filename = f"{row['split']}_{orig_name}"

        dst = os.path.join(out_dir, unique_filename)

        if not os.path.exists(dst):
            shutil.copy2(row["img_path"], dst)
        copied += 1

    actual = len(list(Path(out_dir).iterdir()))
    print(f"  {label}: {actual} images in folder")
    return actual

n_normal   = copy_class(normal_df,   NORMAL_DIR,   "Normal")
n_abnormal = copy_class(abnormal_df, ABNORMAL_DIR, "Abnormal")

Copying Normal: 100%|██████████| 669/669 [00:00<00:00, 2600.43it/s]


  Normal: 669 images in folder


Copying Abnormal: 100%|██████████| 2531/2531 [00:00<00:00, 2598.10it/s]

  Abnormal: 2531 images in folder


# Config

We attempt to apply the same parameter as mentioned in the paper in this notebook.
- resolution = 128: The optimal size for retinal images in this study.
- batch_size = 16: GANs are unstable. Using a small batch size introduces a healthy amount of noise into the training updates.
- best_epoch_quality = 500: The paper found that the visual quality of the synthetic images (measured by the Fréchet Inception Distance, or FID) was best at the very end of training.

In [7]:
from dataclasses import dataclass, field
from pathlib import Path

@dataclass
class DCGANConfig:
    # Architecture (from paper §3.1)
    z_dim       : int   = 100          # Gaussian latent dim — paper uses 100
    resolution  : int   = 128          # 128×128 — paper standard for fundus DCGAN
    base_ch_g   : int   = 64           # generator base channels
    base_ch_d   : int   = 64           # discriminator base channels

    # Training (from paper §3.1)
    batch_size  : int   = 16           # paper: batch size 16
    epochs      : int   = 500          # paper: 500 epochs
    lr          : float = 1e-4         # paper: Adam lr = 0.0001
    beta1       : float = 0.5          # Adam β1 — DCGAN standard
    beta2       : float = 0.999        # Adam β2

    # Data
    normal_dir   : str  = "/content/data/normal"
    abnormal_dir : str  = "/content/data/abnormal"
    num_workers  : int  = 4

    # Generation targets
    # Paper: select images from epoch 500 for quality, epoch 250 for diversity
    normal_target    : int = 2000      # generate normal up to 2000
    abnormal_target  : int = 2000      # generate abnormal up to 2000
    best_epoch_quality    : int = 500  # FID best — paper Table 6
    best_epoch_diversity  : int = 250  # MS-SSIM/CD best — paper Table 6

    # Output
    out_normal   : str  = "/content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/dcgan_normal"
    out_abnormal : str  = "/content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/dcgan_abnormal"
    gen_normal   : str  = "/content/generated/normal"
    gen_abnormal : str  = "/content/generated/abnormal"


cfg = DCGANConfig()

# Set FID sample counts to match real image counts (paper methodology)
cfg.fid_n_normal   = n_normal
cfg.fid_n_abnormal = n_abnormal

for d in [cfg.out_normal, cfg.out_abnormal, cfg.gen_normal, cfg.gen_abnormal]:
    os.makedirs(d, exist_ok=True)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")
print(f"Config  : {cfg.resolution}×{cfg.resolution}  |  z_dim={cfg.z_dim}  "
      f"|  bs={cfg.batch_size}  |  epochs={cfg.epochs}  |  lr={cfg.lr}")
print(f"FID samples — Normal: {cfg.fid_n_normal}  |  Abnormal: {cfg.fid_n_abnormal}")

Device  : cuda
Config  : 128×128  |  z_dim=100  |  bs=16  |  epochs=500  |  lr=0.0001
FID samples — Normal: 669  |  Abnormal: 2531


# Data Loader

In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class FundusDataset(Dataset):
    """
    Single-class flat image folder.
    Normalised to [-1, 1] to match tanh generator output.
    Mild augmentation only — paper uses 128×128 with standard loading.
    """
    def __init__(self, img_dir, resolution=128):
        self.paths = sorted([
            p for p in Path(img_dir).iterdir()
            if p.suffix.lower() in {".png", ".jpg", ".jpeg"}
        ])
        assert len(self.paths) > 0, f"No images in {img_dir}"

        self.transform = transforms.Compose([
            transforms.Resize((resolution, resolution),
                              interpolation=transforms.InterpolationMode.LANCZOS),
            transforms.RandomHorizontalFlip(),   # valid for fundus
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),   # → [-1, 1]
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        return self.transform(Image.open(self.paths[idx]).convert("RGB"))


def make_loader(img_dir, cfg):
    ds = FundusDataset(img_dir, cfg.resolution)
    return DataLoader(
        ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=True,
        persistent_workers=True,
    )

loader_normal   = make_loader(cfg.normal_dir,   cfg)
loader_abnormal = make_loader(cfg.abnormal_dir, cfg)

print(f"Normal   loader: {len(loader_normal.dataset)} images  "
      f"| {len(loader_normal)} batches/epoch")
print(f"Abnormal loader: {len(loader_abnormal.dataset)} images  "
      f"| {len(loader_abnormal)} batches/epoch")

Normal   loader: 669 images  | 41 batches/epoch
Abnormal loader: 2531 images  | 158 batches/epoch


# DCGAN Architecture

This implementation improves upon the standard DCGAN from the paper by introducing a stabilization techniques:
- Spectral Normalization (https://arxiv.org/pdf/1802.05957): By wrapping every convolutional layer (nn.Conv2d) in spectral_norm, the model artificially constrains how large the weights can grow (Lipschitz constraint). This keeps the discriminator's power in check and prevents it from easily overpowering the generator, a common cause of GAN collapse.

Loss Function and Stability
- Binary Cross-Entropy (BCE) Loss: The standard objective for GANs. Working alongside the final Sigmoid activation.
- Label Smoothing: Instead of using absolute targets (1.0 for real, 0.0 for fake), this technique softens the targets (e.g., telling the discriminator a real image is 0.9). This prevents the discriminator from becoming 100% confident too early.

In [9]:
# Paper §3.1: redesigned with deconvolution/convolution layers ONLY
# "layers were redesigned with deconvolution and convolution layers only"
# to avoid gradient explosion from BatchNorm + upsampling combinations
# Generator : 4 deconvolutional layers
# Discriminator: 5 convolutional layers

import torch.nn as nn
import math

def weights_init(m):
    """DCGAN standard: Conv/Linear weights ~ N(0, 0.02)."""
    cls = m.__class__.__name__
    if "Conv" in cls:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0)


class DCGANGenerator(nn.Module):
    """
    4 deconvolutional layers — no BatchNorm (paper §3.1).
    z (100,) → RGB (3, 128, 128)
    Layer progression: z→(512,4,4)→(256,8,8)→(128,16,16)→(64,32,32)
                       →(32,64,64)→(3,128,128)
    """
    def __init__(self, z_dim=100, base_ch=64, resolution=128):
        super().__init__()
        n_up   = int(math.log2(resolution)) - 2   # 128 → 5 upsamples
        ch_max = base_ch * (2 ** (n_up - 1))       # 64 * 16 = 1024, cap at 512
        ch_max = min(ch_max, 512)

        def ch(stage):
            return max(base_ch, min(512, base_ch * (2 ** (n_up - 1 - stage))))

        # Stage 0: z → 4×4
        layers = [
            nn.ConvTranspose2d(z_dim, ch(0), 4, 1, 0, bias=True),
            nn.ReLU(True),
        ]
        # Upsample stages
        for stage in range(n_up - 1):
            layers += [
                nn.ConvTranspose2d(ch(stage), ch(stage + 1), 4, 2, 1, bias=True),
                nn.ReLU(True),
            ]
        # Final → RGB
        layers += [
            nn.ConvTranspose2d(ch(n_up - 1), 3, 4, 2, 1, bias=True),
            nn.Tanh(),
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z.view(z.size(0), -1, 1, 1))


# class DCGANDiscriminator(nn.Module):
#     """
#     5 convolutional layers — no BatchNorm (paper §3.1).
#     RGB (3, 128, 128) → scalar
#     """
#     def __init__(self, base_ch=64, resolution=128):
#         super().__init__()
#         n_down = int(math.log2(resolution)) - 2   # 128 → 5 downsamples

#         def ch(stage):
#             return min(512, base_ch * (2 ** stage))

#         # First layer — no norm
#         layers = [
#             nn.Conv2d(3, ch(0), 4, 2, 1, bias=True),
#             nn.LeakyReLU(0.2, inplace=True),
#         ]
#         for stage in range(n_down - 1):
#             layers += [
#                 nn.Conv2d(ch(stage), ch(stage + 1), 4, 2, 1, bias=True),
#                 nn.LeakyReLU(0.2, inplace=True),
#             ]
#         # Final → scalar
#         layers += [
#             nn.Conv2d(ch(n_down - 1), 1, 4, 1, 0, bias=True),
#             nn.Sigmoid(),
#         ]
#         self.net = nn.Sequential(*layers)

#     def forward(self, x):
#         return self.net(x).view(-1)

class DCGANDiscriminator(nn.Module):
    def __init__(self, base_ch=64, resolution=128):
        super().__init__()
        import math
        n_down = int(math.log2(resolution)) - 2

        def ch(stage):
            return min(512, base_ch * (2 ** stage))

        layers = [
            # ← wrap every Conv2d with spectral_norm
            nn.utils.spectral_norm(
                nn.Conv2d(3, ch(0), 4, 2, 1, bias=True)
            ),
            nn.LeakyReLU(0.2, inplace=True),
        ]
        for stage in range(n_down - 1):
            layers += [
                nn.utils.spectral_norm(
                    nn.Conv2d(ch(stage), ch(stage+1), 4, 2, 1, bias=True)
                ),
                nn.LeakyReLU(0.2, inplace=True),
            ]
        layers += [
            nn.utils.spectral_norm(
                nn.Conv2d(ch(n_down-1), 1, 4, 1, 0, bias=True)
            ),
            nn.Sigmoid(),    # ← keep sigmoid, keep BCE
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).view(-1)


def build_models(cfg, device):
    G = DCGANGenerator(z_dim=cfg.z_dim, base_ch=cfg.base_ch_g,
                       resolution=cfg.resolution).to(device)
    D = DCGANDiscriminator(base_ch=cfg.base_ch_d,
                           resolution=cfg.resolution).to(device)
    G.apply(weights_init)
    D.apply(weights_init)
    return G, D


# Sanity check
G_test, D_test = build_models(cfg, device)
z = torch.randn(2, cfg.z_dim, device=device)
with torch.no_grad():
    img   = G_test(z)
    score = D_test(img)

print(f"G output : {img.shape}")     # (2, 3, 128, 128)
print(f"D output : {score.shape}")   # (2,)
print(f"G params : {sum(p.numel() for p in G_test.parameters()):,}")
print(f"D params : {sum(p.numel() for p in D_test.parameters()):,}")
del G_test, D_test

G output : torch.Size([2, 3, 128, 128])
D output : torch.Size([2])
G params : 7,770,563
D params : 6,959,553


# Training Function

In [10]:
import torch.nn as nn
import torch.optim as optim
from torchvision.utils import save_image
import json, glob, time, os
from pathlib import Path

def train_dcgan(loader, out_dir, class_name, cfg, device,
                resume=True, save_every=100, sample_every=25):
    """
    Trains one DCGAN for one class.
    Saves checkpoints every save_every epochs.
    Saves fixed-noise sample grids every sample_every epochs.
    Resumes from latest checkpoint if resume=True.
    Returns trained G.
    """
    os.makedirs(out_dir, exist_ok=True)

    G, D = build_models(cfg, device)

    criterion = nn.BCELoss()
    # Paper: Adam, lr=0.0001, β1=0.5, β2=0.999
    opt_G = optim.Adam(G.parameters(), lr=cfg.lr,
                       betas=(cfg.beta1, cfg.beta2))
    opt_D = optim.Adam(D.parameters(), lr=cfg.lr,
                       betas=(cfg.beta1, cfg.beta2))

    history = {"epoch": [], "loss_G": [], "loss_D": [], "D_x": [], "D_Gz": []}
    start_epoch = 1

    # Resume
    if resume:
        ckpts = sorted(glob.glob(os.path.join(out_dir, "ckpt_epoch_*.pt")))
        if ckpts:
            ckpt = torch.load(ckpts[-1], map_location=device)
            G.load_state_dict(ckpt["G_state"])
            D.load_state_dict(ckpt["D_state"])
            opt_G.load_state_dict(ckpt["opt_G"])
            opt_D.load_state_dict(ckpt["opt_D"])
            start_epoch = ckpt["epoch"] + 1
            history     = ckpt["history"]
            print(f"[{class_name}] Resumed from epoch {ckpt['epoch']}")
        else:
            print(f"[{class_name}] Starting from scratch")

    fixed_z = torch.randn(16, cfg.z_dim, device=device)
    LABEL_SMOOTH = 0.9    # soft real labels for stability

    # Training loop
    for epoch in range(start_epoch, cfg.epochs + 1):
        G.train(); D.train()
        t0 = time.time()
        e_lG = e_lD = e_Dx = e_DGz = 0.0

        for real_imgs in loader:
            real_imgs = real_imgs.to(device)
            B = real_imgs.size(0)

            real_labels = torch.full((B,), LABEL_SMOOTH, device=device)
            fake_labels = torch.zeros(B, device=device)

            # D step
            opt_D.zero_grad()
            out_real    = D(real_imgs)
            loss_D_real = criterion(out_real, real_labels)
            D_x         = out_real.mean().item()

            z         = torch.randn(B, cfg.z_dim, device=device)
            with torch.no_grad():
                fake_imgs = G(z)
            out_fake    = D(fake_imgs)
            loss_D_fake = criterion(out_fake, fake_labels)

            loss_D = loss_D_real + loss_D_fake
            loss_D.backward()
            opt_D.step()

            # G step
            opt_G.zero_grad()
            z         = torch.randn(B, cfg.z_dim, device=device)
            fake_imgs = G(z)
            out_fake  = D(fake_imgs)
            loss_G    = criterion(out_fake, real_labels)
            loss_G.backward()
            opt_G.step()

            D_Gz  = out_fake.mean().item()
            e_lG += loss_G.item()
            e_lD += loss_D.item()
            e_Dx += D_x
            e_DGz += D_Gz

        # Epoch end
        n = len(loader)
        history["epoch"].append(epoch)
        history["loss_G"].append(e_lG / n)
        history["loss_D"].append(e_lD / n)
        history["D_x"].append(e_Dx / n)
        history["D_Gz"].append(e_DGz / n)

        elapsed = time.time() - t0
        if epoch % 50 == 0 or epoch == cfg.epochs - 1:
          print(f"[{class_name}] Ep {epoch:03d}/{cfg.epochs}  "
                f"L_D {e_lD/n:.4f}  L_G {e_lG/n:.4f}  "
                f"D(x) {e_Dx/n:.3f}  D(G(z)) {e_DGz/n:.3f}  "
                f"({elapsed:.0f}s)")

        # Sample grid
        if epoch % sample_every == 0:
            G.eval()
            with torch.no_grad():
                samples = G(fixed_z)
            samples = (samples * 0.5 + 0.5).clamp(0, 1)
            save_image(samples,
                       os.path.join(out_dir, f"sample_ep{epoch:04d}.png"),
                       nrow=4)

        # Checkpoint
        if epoch % save_every == 0 or epoch == cfg.epochs:
            ckpt_path = os.path.join(out_dir, f"ckpt_epoch_{epoch:04d}.pt")
            torch.save({
                "epoch"  : epoch,
                "G_state": G.state_dict(),
                "D_state": D.state_dict(),
                "opt_G"  : opt_G.state_dict(),
                "opt_D"  : opt_D.state_dict(),
                "history": history,
            }, ckpt_path)
            print(f"  → Checkpoint: {ckpt_path}")

    with open(os.path.join(out_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)

    print(f"\n[{class_name}] Training complete ✓")
    return G, history

# Train Both DCGANs

- A standard DCGAN is an unconditional generative model. It does not accept class labels as input, so we must train two separate models, one exclusively for normal images and one for abnormal images.

In [11]:
# Run sequentially — one GPU, one class at a time

print("=" * 60)
print("DCGAN 1/2 — Normal fundus")
print("=" * 60)
G_normal, hist_normal = train_dcgan(
    loader     = loader_normal,
    out_dir    = cfg.out_normal,
    class_name = "Normal",
    cfg        = cfg,
    device     = device,
)

print("\n" + "=" * 60)
print("DCGAN 2/2 — Abnormal fundus")
print("=" * 60)
G_abnormal, hist_abnormal = train_dcgan(
    loader     = loader_abnormal,
    out_dir    = cfg.out_abnormal,
    class_name = "Abnormal",
    cfg        = cfg,
    device     = device,
)

DCGAN 1/2 — Normal fundus
[Normal] Starting from scratch
[Normal] Ep 050/500  L_D 0.9652  L_G 1.6888  D(x) 0.641  D(G(z)) 0.189  (3s)
[Normal] Ep 100/500  L_D 0.8482  L_G 1.9154  D(x) 0.665  D(G(z)) 0.153  (3s)
  → Checkpoint: /content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/dcgan_normal/ckpt_epoch_0100.pt
[Normal] Ep 150/500  L_D 0.9333  L_G 1.5213  D(x) 0.624  D(G(z)) 0.225  (3s)
[Normal] Ep 200/500  L_D 1.0177  L_G 1.3174  D(x) 0.589  D(G(z)) 0.276  (3s)
  → Checkpoint: /content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/dcgan_normal/ckpt_epoch_0200.pt
[Normal] Ep 250/500  L_D 1.0807  L_G 1.2646  D(x) 0.567  D(G(z)) 0.286  (3s)
[Normal] Ep 300/500  L_D 1.0583  L_G 1.2097  D(x) 0.574  D(G(z)) 0.303  (3s)
  → Checkpoint: /content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/dcgan_normal/ckpt_epoch_0300.pt
[Normal] Ep 350/500  L_D 1.0745  L_G 1.2760  D(x) 0.569  D(G(z)) 0.282  (3s)
[Normal] Ep 400/500  L_D 1.0452  L_G 1.2805  D(x) 0.577  D(G(z)) 0.283  (3s)
  →

- Each training session lasts approximately 1 hour.


# Generate synthetic images

- We will generate a normal and abnormal class with each 500 samples, totaling 1,000 samples. However, in this experiment, we unintentionally set it to each for 2,000 samples, but we only chose 500 for each in the evaluation.

In [12]:
# Paper §4.2.3: select epoch 500 for quality (best FID)

from PIL import Image

def load_G_from_epoch(out_dir, epoch, cfg, device):
    """Load generator weights from a specific epoch checkpoint."""
    pattern = os.path.join(out_dir, f"ckpt_epoch_{epoch:04d}.pt")
    matches = glob.glob(pattern)
    if not matches:
        # Fall back to latest
        matches = sorted(glob.glob(os.path.join(out_dir, "ckpt_epoch_*.pt")))
        if not matches:
            raise FileNotFoundError(f"No checkpoints in {out_dir}")
        print(f"  Epoch {epoch} not found — using latest: {matches[-1]}")
        ckpt_path = matches[-1]
    else:
        ckpt_path = matches[0]

    G, _ = build_models(cfg, device)
    ckpt  = torch.load(ckpt_path, map_location=device)
    G.load_state_dict(ckpt["G_state"])
    G.eval()
    print(f"  Loaded: {ckpt_path}")
    return G


def generate_images(G, out_dir, n_images, class_name,
                    seed=42, batch=32):
    """
    Generate n_images synthetic images from trained G.
    Paper: generates from random z — pure unconditional sampling.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(seed)
    generated = 0

    with torch.no_grad():
        while generated < n_images:
            bs = min(batch, n_images - generated)
            z  = torch.randn(bs, cfg.z_dim, device=device)
            imgs = G(z)
            # [-1,1] → [0,255] uint8
            imgs = ((imgs * 0.5 + 0.5) * 255) \
                       .clamp(0, 255) \
                       .permute(0, 2, 3, 1) \
                       .to(torch.uint8) \
                       .cpu().numpy()
            for j, img in enumerate(imgs):
                Image.fromarray(img).save(
                    out_dir / f"{class_name}_gen_{generated+j:05d}.png"
                )
            generated += bs

    print(f"  [{class_name}] Generated {generated} images → {out_dir}")


# Generate from epoch 500 (quality-optimised — best FID per paper Table 6)
print("Generating from epoch 500 (quality-optimised)...")

G_n_500 = load_G_from_epoch(cfg.out_normal,   cfg.best_epoch_quality, cfg, device)
generate_images(G_n_500, cfg.gen_normal + "_ep500",
                cfg.normal_target,   "normal")

G_a_500 = load_G_from_epoch(cfg.out_abnormal, cfg.best_epoch_quality, cfg, device)
generate_images(G_a_500, cfg.gen_abnormal + "_ep500",
                cfg.abnormal_target, "abnormal")

print("\nGeneration complete ✓")

Generating from epoch 500 (quality-optimised)...
  Loaded: /content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/dcgan_normal/ckpt_epoch_0500.pt
  [normal] Generated 2000 images → /content/generated/normal_ep500
  Loaded: /content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/dcgan_abnormal/ckpt_epoch_0500.pt
  [abnormal] Generated 2000 images → /content/generated/abnormal_ep500

Generation complete ✓


In [18]:
# import shutil
# from google.colab import files

# shutil.make_archive('/content/normal_images', 'zip', '/content/generated/normal_ep500')
# shutil.make_archive('/content/abnormal_images', 'zip', '/content/generated/abnormal_ep500')

'/content/abnormal_images.zip'

In [19]:
# files.download('/content/normal_images.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
# files.download('/content/abnormal_images.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Evaluation

- **FID (Fréchet Inception Distance):** evaluates the overall quality and realism of your synthetic retinal images. A low FID score means the statistical distribution of features (like color, texture, and structural layout) in generated images closely matches the distribution of the real RFMiD dataset.

- **SSIM (Structural Similarity Index Measure):** calculates the structural, luminance, and contrast similarity between paired images.

- **PSNR (Peak Signal-to-Noise Ratio):** measures the ratio between the maximum possible power of a signal (the clean image data) and the corrupting noise affecting its representation.

- Since the dataset we got was larger than 128x128, we resized the actual images to match the size of the generated images during the evaluation process.
- For the pairing evaluation for SSIM/PSNR, we simply create the exact format generated images set because we have already prepared the evaluation set and imported it. The first 500 images are normal class, and the remaining 500 are abnormal class.

In [9]:
def count_images(d):
    return len(glob.glob(os.path.join(d, "*.png")) +
               glob.glob(os.path.join(d, "*.jpg")))

def copy_n_sorted(src_dir, dst_dir, n, prefix, seed=42):
    """
    Copy exactly n images from src_dir into dst_dir.
    Named with prefix + index so sort order is guaranteed.
    """
    paths = sorted(
        glob.glob(os.path.join(src_dir, "*.png")) +
        glob.glob(os.path.join(src_dir, "*.jpg"))
    )
    random.seed(seed)
    selected = random.sample(paths, min(n, len(paths)))
    # Re-sort after sampling so filenames are deterministic
    selected = sorted(selected)

    for i, p in enumerate(selected):
        ext = os.path.splitext(p)[1]
        # prefix ensures normal comes before abnormal in sort order
        shutil.copy2(p, os.path.join(dst_dir, f"{prefix}_{i:05d}{ext}"))

    print(f"  Copied {len(selected)} {prefix} → {dst_dir}")
    return len(selected)

def calculate_metrics(real_dir, fake_dir, device, n_samples,
                      batch_size=32, seed=42, preserve_order=False):
    # FID
    try:
        fid = fid_score.calculate_fid_given_paths(
            [real_dir, fake_dir],
            batch_size=batch_size,
            device=device,
            dims=2048,
        )
    except Exception as e:
        print(f"  FID error: {e}")
        fid = None

    # SSIM & PSNR
    real_paths = sorted(
        glob.glob(os.path.join(real_dir, "*.png")) +
        glob.glob(os.path.join(real_dir, "*.jpg"))
    )
    fake_paths = sorted(
        glob.glob(os.path.join(fake_dir, "*.png")) +
        glob.glob(os.path.join(fake_dir, "*.jpg"))
    )

    n_pairs = min(len(real_paths), len(fake_paths), n_samples)

    if preserve_order:
        # Keep positional pairing
        # Just truncate to n_pairs, no shuffling
        real_paths = real_paths[:n_pairs]
        fake_paths = fake_paths[:n_pairs]
    else:
        # Original random sampling — for when pairing doesn't matter
        random.seed(seed)
        real_paths = random.sample(real_paths, n_pairs)
        fake_paths = random.sample(fake_paths, n_pairs)

    ssim_scores = []
    psnr_scores = []
    skipped     = 0

    for real_path, fake_path in zip(real_paths, fake_paths):
        real_img = cv2.imread(real_path, cv2.IMREAD_GRAYSCALE)
        fake_img = cv2.imread(fake_path, cv2.IMREAD_GRAYSCALE)

        if real_img is None or fake_img is None:
            skipped += 1
            continue

        # Resize real DOWN to 128 to match GAN output
        if real_img.shape != (GAN_RES, GAN_RES):
            real_img = cv2.resize(
                real_img,
                (GAN_RES, GAN_RES),
                interpolation=cv2.INTER_LANCZOS4,
            )

        # Safety check on gen size
        if fake_img.shape != (GAN_RES, GAN_RES):
            fake_img = cv2.resize(
                fake_img,
                (GAN_RES, GAN_RES),
                interpolation=cv2.INTER_LANCZOS4,
            )

        assert real_img.shape == fake_img.shape, \
            f"Shape mismatch: {real_img.shape} vs {fake_img.shape}"

        ssim_scores.append(ssim(real_img, fake_img, data_range=255))
        psnr_scores.append(psnr(real_img, fake_img, data_range=255))

    if skipped > 0:
        print(f"  Skipped {skipped} pairs due to read errors")

    avg_ssim = float(np.mean(ssim_scores)) if ssim_scores else -1.0
    avg_psnr = float(np.mean(psnr_scores)) if psnr_scores else -1.0

    return fid, avg_ssim, avg_psnr

In [10]:
import numpy as np
import glob, os, shutil, random
import cv2
import torch
import json

import subprocess
subprocess.run("pip install -q pytorch-fid", shell=True, check=True)

from pytorch_fid import fid_score
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from pathlib import Path

In [30]:
ZIP_NORMAL   = "/content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/dcgan_normal.zip"
ZIP_ABNORMAL = "/content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/dcgan_abnormal.zip"
ZIP_EVAL     = "/content/drive/MyDrive/Colab Notebooks/CPE494_GenAI/project/eval_img.zip"

GEN_NORMAL_DIR   = "/content/eval/gen_normal"
GEN_ABNORMAL_DIR = "/content/eval/gen_abnormal"
EVAL_REAL_DIR    = "/content/eval/real"
COMBINED_GEN_DIR = "/content/eval/combined_gen"

GAN_RES = 128
SEED    = 42
N_EACH  = 500
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"

In [31]:
for d in [GEN_NORMAL_DIR, GEN_ABNORMAL_DIR, EVAL_REAL_DIR, COMBINED_GEN_DIR]:
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d)

In [32]:
print("Unzipping...")
shutil.unpack_archive(ZIP_NORMAL,   GEN_NORMAL_DIR)
shutil.unpack_archive(ZIP_ABNORMAL, GEN_ABNORMAL_DIR)
shutil.unpack_archive(ZIP_EVAL,     EVAL_REAL_DIR)

Unzipping...


In [35]:
EVAL_IMG_DIR = "/content/eval/real/images"

In [36]:
print(f"  gen normal      : {count_images(GEN_NORMAL_DIR)}")
print(f"  gen abnormal    : {count_images(GEN_ABNORMAL_DIR)}")
print(f"  eval real total : {count_images(EVAL_IMG_DIR)}")

  gen normal      : 2000
  gen abnormal    : 2000
  eval real total : 1000


In [37]:
print("\nBuilding combined_gen (normal first, abnormal second)...")
n_gen_normal   = copy_n_sorted(GEN_NORMAL_DIR,   COMBINED_GEN_DIR,
                                N_EACH, "a_normal",   SEED)
n_gen_abnormal = copy_n_sorted(GEN_ABNORMAL_DIR, COMBINED_GEN_DIR,
                                N_EACH, "b_abnormal", SEED)

print(f"  combined_gen total : {count_images(COMBINED_GEN_DIR)}")


Building combined_gen (normal first, abnormal second)...
  Copied 500 a_normal → /content/eval/combined_gen
  Copied 500 b_abnormal → /content/eval/combined_gen
  combined_gen total : 1000


In [38]:
real_paths = sorted(
    glob.glob(os.path.join(EVAL_IMG_DIR,    "*.png")) +
    glob.glob(os.path.join(EVAL_IMG_DIR,    "*.jpg"))
)
gen_paths  = sorted(
    glob.glob(os.path.join(COMBINED_GEN_DIR, "*.png")) +
    glob.glob(os.path.join(COMBINED_GEN_DIR, "*.jpg"))
)

assert len(real_paths) == len(gen_paths), \
    f"Count mismatch: {len(real_paths)} real vs {len(gen_paths)} gen"

print(f"Pairs ready: {len(real_paths)} ✓")

Pairs ready: 1000 ✓


In [39]:
print("Computing metrics...")
fid_val, avg_ssim, avg_psnr = calculate_metrics(
    real_dir       = EVAL_IMG_DIR,
    fake_dir       = COMBINED_GEN_DIR,
    device         = DEVICE,
    n_samples      = N_EACH * 2,    # 1000 total pairs
    batch_size     = 32,
    seed           = SEED,
    preserve_order = True,          # ← keeps curated order
)

Computing metrics...
Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth


100%|██████████| 91.2M/91.2M [00:00<00:00, 498MB/s]
100%|██████████| 32/32 [00:01<00:00, 20.14it/s]


In [40]:
print(f"""
┌──────────────────────────┬──────────────┬──────────────┬──────────────┐
│ Evaluation               │     FID ↓    │    SSIM ↑    │    PSNR ↑    │
├──────────────────────────┼──────────────┼──────────────┼──────────────┤
│ Combined (1000 pairs)    │ {fid_val:12.4f} │ {avg_ssim:12.4f} │ {avg_psnr:12.4f} │
└──────────────────────────┴──────────────┴──────────────┴──────────────┘
FID  : lower is better  — 1000 real vs 1000 generated
SSIM : higher is better — 1000 positional pairs (500 normal + 500 abnormal)
PSNR : higher is better — 1000 positional pairs (500 normal + 500 abnormal)
""")


┌──────────────────────────┬──────────────┬──────────────┬──────────────┐
│ Evaluation               │     FID ↓    │    SSIM ↑    │    PSNR ↑    │
├──────────────────────────┼──────────────┼──────────────┼──────────────┤
│ Combined (1000 pairs)    │     114.2975 │       0.5080 │      16.7365 │
└──────────────────────────┴──────────────┴──────────────┴──────────────┘
FID  : lower is better  — 1000 real vs 1000 generated
SSIM : higher is better — 1000 positional pairs (500 normal + 500 abnormal)
PSNR : higher is better — 1000 positional pairs (500 normal + 500 abnormal)



Discussion
- From the evaluation result, it shows that DCGAN establishes the baseline with an FID of 114.30, SSIM of 0.51, and PSNR of 16.74.
- The DCGAN successfully learned the macroscopic features of the retina such as the optic disc and background pigmentation but struggled to generate sharp, high-frequency details like capillary networks due to its 128×128 resolution limits.
- Despite these resolution constraints, the training logs confirm that implementing spectral normalization and label smoothing successfully stabilized the model. The discriminator and generator maintained a healthy adversarial dynamic throughout 500 epochs, effectively preventing catastrophic mode collapse.